# E-Commerce Marketplace Analytics — Data Cleaning

## Project Goal
This notebook prepares the Brazilian E-Commerce Public Dataset (Olist) for the
**E-Commerce Marketplace Analytics: Sales, Customer & Delivery Performance**
project. It is the first notebook in the pipeline and produces the clean,
analysis-ready data that every later notebook (SQLite database construction, SQL
analysis, Python analysis, and visualization) depends on.

## Dataset
The Brazilian E-Commerce Public Dataset by Olist
(https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — a real,
anonymized dataset of ~100,000 orders placed on the Olist marketplace in
Brazil between 2016 and 2018. This project uses six of the nine files Olist
publishes:

- `olist_orders_dataset.csv`
- `olist_order_items_dataset.csv`
- `olist_customers_dataset.csv`
- `olist_products_dataset.csv`
- `product_category_name_translation.csv`
- `olist_order_reviews_dataset.csv`

The `sellers`, `order_payments`, and `geolocation` files are intentionally
**not** loaded here — they are optional/bonus extensions and are not
required for the core six-table schema.

## Why Cleaning Is Required
Real transactional data always has structural quirks. In this dataset,
specifically:
- Customers are assigned a new `customer_id` for every order, so a true
  customer identity requires deduplicating on `customer_unique_id` instead.
- Product categories are recorded in Portuguese, and a small number of
  category names have no match in the official English translation file.
- Timestamps, foreign keys, and row-level duplication all need to be
  validated before any KPI or SQL work can be trusted.

## What This Notebook Accomplishes
1. Loads all six core raw files and profiles them.
2. Runs a full set of data quality checks against them.
3. Cleans the data using clearly documented, non-destructive rules —
   **no row is ever silently dropped** without a logged reason and count.
4. Exports six clean, analysis-ready CSVs to `data/processed/`.
5. Generates `reports/data_quality_report.md`, a Markdown report built
   directly from this notebook's own validation results.

This notebook does **not** build the SQLite database, run SQL, perform EDA,
calculate KPIs, or create charts — those happen in the notebooks that follow.


## Section 2 — Import Libraries

Only the libraries actually needed for cleaning are imported: `pandas` for
tabular data handling, `numpy` for numeric operations, `pathlib` for
filesystem-independent paths, and `warnings` to keep output focused on this
notebook's own messages rather than library noise.


In [2]:
from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## Section 3 — Load Data

All six core raw files are loaded from `data/raw/`. If a file is missing,
loading fails immediately with a clear error rather than continuing with
partial data — place the real Olist CSVs in `data/raw/` before running this
notebook.


In [3]:
# Paths are relative to this notebook's location inside notebooks/.
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES: Dict[str, str] = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "reviews": "olist_order_reviews_dataset.csv",
}

TIMESTAMP_COLUMNS: Dict[str, List[str]] = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}


def load_raw_datasets(raw_dir: Path, file_map: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    """Load every raw CSV listed in file_map into a dict of DataFrames.

    Raises FileNotFoundError immediately if any expected file is missing,
    rather than silently proceeding with partial data.
    """
    datasets: Dict[str, pd.DataFrame] = {}
    for key, filename in file_map.items():
        file_path = raw_dir / filename
        if not file_path.exists():
            raise FileNotFoundError(
                f"Expected raw file not found: {file_path}\n"
                "Place the official Olist CSV files in data/raw/ before running this notebook."
            )
        datasets[key] = pd.read_csv(file_path)
    return datasets


raw = load_raw_datasets(RAW_DIR, RAW_FILES)

for name, df in raw.items():
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    print(df.head(3))
    print()


--- orders ---
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
                           order_id                       customer_id order_status order_purchase_timestamp  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49   

     order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date  
0  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00  
1  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45           20

In [4]:
# Full .info() output per dataset, shown separately for readability.
for name, df in raw.items():
    print(f"=== {name}.info() ===")
    df.info()
    print()


=== orders.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

=== order_items.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  


## Section 4 — Initial Data Audit

A structured, reusable audit is run across every dataset: row/column counts,
memory footprint, duplicate rows, and total missing cells. This gives a
single at-a-glance summary before any deeper quality checks.


In [5]:
def summarize_dataset(name: str, df: pd.DataFrame) -> pd.Series:
    """Return a one-row shape/memory/missingness/duplicate summary for a DataFrame."""
    return pd.Series(
        {
            "dataset": name,
            "rows": len(df),
            "columns": df.shape[1],
            "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 3),
            "duplicate_rows": int(df.duplicated().sum()),
            "total_missing_cells": int(df.isnull().sum().sum()),
        }
    )


audit_summary = pd.DataFrame([summarize_dataset(name, df) for name, df in raw.items()])
audit_summary


,dataset,rows,columns,memory_mb,duplicate_rows,total_missing_cells
0,orders,99441,8,52.937,0,4908
1,order_items,112650,7,35.990,0,0
2,customers,99441,5,26.586,0,0
3,products,32951,9,6.297,0,2448
4,category_translation,71,2,0.009,0,0
5,reviews,99224,7,39.125,0,145903


In [6]:
def missingness_table(name: str, df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-column missing value count and percentage for one dataset."""
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        return pd.DataFrame(columns=["dataset", "column", "missing_count", "missing_pct"])
    pct = (missing / len(df) * 100).round(2)
    out = pd.DataFrame(
        {
            "dataset": name,
            "column": missing.index,
            "missing_count": missing.values,
            "missing_pct": pct.values,
        }
    )
    return out.sort_values("missing_pct", ascending=False).reset_index(drop=True)


missingness = pd.concat(
    [missingness_table(name, df) for name, df in raw.items()], ignore_index=True
)
missingness


,dataset,column,missing_count,missing_pct
0,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


In [7]:
# Unique value counts per column, useful for spotting categorical inconsistencies
# (e.g., inconsistent casing in city names) before cleaning.
for name, df in raw.items():
    print(f"--- {name}: unique values per column ---")
    print(df.nunique())
    print()


--- orders: unique values per column ---
order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_customer_date    95664
order_estimated_delivery_date      459
dtype: int64

--- order_items: unique values per column ---
order_id               98666
order_item_id             21
product_id             32951
seller_id               3095
shipping_limit_date    93318
price                   5968
freight_value           6999
dtype: int64

--- customers: unique values per column ---
customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

--- products: unique values per column ---
product_id                    32951
product_category_name            73
product_name_lenght        

## Section 5 — Data Quality Assessment

Each data quality rule below is implemented as a small, reusable function,
then applied to the relevant tables. Nothing is changed in this section —
only detected and documented.


In [8]:
def check_primary_key_uniqueness(name: str, df: pd.DataFrame, key_cols: List[str]) -> Dict[str, object]:
    """Check whether the given key column(s) uniquely identify rows in df."""
    n_total = len(df)
    n_unique = df.drop_duplicates(subset=key_cols).shape[0]
    return {
        "dataset": name,
        "key_columns": ", ".join(key_cols),
        "total_rows": n_total,
        "unique_key_rows": n_unique,
        "duplicate_key_rows": n_total - n_unique,
        "is_unique": n_total == n_unique,
    }


pk_checks = [
    check_primary_key_uniqueness("orders", raw["orders"], ["order_id"]),
    check_primary_key_uniqueness("customers", raw["customers"], ["customer_id"]),
    check_primary_key_uniqueness("products", raw["products"], ["product_id"]),
    check_primary_key_uniqueness("reviews", raw["reviews"], ["review_id"]),
]
pk_df = pd.DataFrame(pk_checks)
pk_df


,dataset,key_columns,total_rows,unique_key_rows,duplicate_key_rows,is_unique
0,orders,order_id,99441,99441,0,True
1,customers,customer_id,99441,99441,0,True
2,products,product_id,32951,32951,0,True
3,reviews,review_id,99224,98410,814,False


In [9]:
def check_foreign_key(
    child_name: str,
    child_df: pd.DataFrame,
    child_col: str,
    parent_name: str,
    parent_df: pd.DataFrame,
    parent_col: str,
) -> Dict[str, object]:
    """Check how many child rows reference a value absent from the parent table."""
    valid_keys = set(parent_df[parent_col].dropna().unique())
    child_values = child_df[child_col]
    is_orphan = ~child_values.isin(valid_keys) & child_values.notna()
    n_orphans = int(is_orphan.sum())
    return {
        "relationship": f"{child_name}.{child_col} -> {parent_name}.{parent_col}",
        "total_child_rows": len(child_df),
        "orphan_rows": n_orphans,
        "orphan_pct": round(n_orphans / len(child_df) * 100, 3) if len(child_df) else 0.0,
        "sample_orphan_values": child_values[is_orphan].unique()[:5].tolist(),
    }


fk_checks = [
    check_foreign_key("order_items", raw["order_items"], "order_id", "orders", raw["orders"], "order_id"),
    check_foreign_key(
        "order_items", raw["order_items"], "product_id", "products", raw["products"], "product_id"
    ),
    check_foreign_key("orders", raw["orders"], "customer_id", "customers", raw["customers"], "customer_id"),
    check_foreign_key("reviews", raw["reviews"], "order_id", "orders", raw["orders"], "order_id"),
]
fk_df = pd.DataFrame(fk_checks)
fk_df


,relationship,total_child_rows,orphan_rows,orphan_pct,sample_orphan_values
0,order_items.order_id -> orders.order_id,112650,0,0.0,[]
1,order_items.product_id -> products.product_id,112650,0,0.0,[]
2,orders.customer_id -> customers.customer_id,99441,0,0.0,[]
3,reviews.order_id -> orders.order_id,99224,0,0.0,[]


In [10]:
def check_translation_coverage(products: pd.DataFrame, translation: pd.DataFrame) -> Dict[str, object]:
    """Identify product categories with no matching entry in the English translation file."""
    product_categories = set(products["product_category_name"].dropna().unique())
    translated_categories = set(translation["product_category_name"].dropna().unique())
    untranslated = sorted(product_categories - translated_categories)
    return {
        "total_distinct_categories": len(product_categories),
        "translated_categories": len(product_categories) - len(untranslated),
        "untranslated_categories": len(untranslated),
        "untranslated_category_names": untranslated,
    }


translation_check = check_translation_coverage(raw["products"], raw["category_translation"])
translation_check


{'total_distinct_categories': 73,
 'translated_categories': 71,
 'untranslated_categories': 2,
 'untranslated_category_names': ['pc_gamer',
  'portateis_cozinha_e_preparadores_de_alimentos']}

In [11]:
duplicate_rows_report = pd.DataFrame(
    [{"dataset": name, "exact_duplicate_rows": int(df.duplicated().sum())} for name, df in raw.items()]
)
duplicate_rows_report


,dataset,exact_duplicate_rows
0,orders,0
1,order_items,0
2,customers,0
3,products,0
4,category_translation,0
5,reviews,0


In [12]:
def check_logical_date_order(orders: pd.DataFrame) -> pd.DataFrame:
    """Flag orders where the customer-delivered date is earlier than the purchase date."""
    purchase = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    delivered = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")
    mask = delivered.notna() & purchase.notna() & (delivered < purchase)
    result = orders.loc[mask, ["order_id"]].copy()
    result["order_purchase_timestamp"] = purchase[mask]
    result["order_delivered_customer_date"] = delivered[mask]
    return result


logical_date_issues = check_logical_date_order(raw["orders"])
logical_date_issues


,order_id,order_purchase_timestamp,order_delivered_customer_date


In [13]:
def check_review_cardinality(reviews: pd.DataFrame) -> pd.DataFrame:
    """Identify orders that have more than one review (the schema assumes exactly one)."""
    counts = reviews.groupby("order_id").size()
    multi_order_ids = counts[counts > 1].index
    return reviews[reviews["order_id"].isin(multi_order_ids)].sort_values("order_id")


review_cardinality_issues = check_review_cardinality(raw["reviews"])
review_cardinality_issues


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
...,...,...,...,...,...,...,...
27465,5e78482ee783451be6026e5cf0c72de1,ff763b73e473d03c321bcd5a053316e8,3,NaN,Não sei que haverá acontecido os demais chegaram,2017-11-18 00:00:00,2017-11-18 09:02:48
41355,39de8ad3a1a494fc68cc2d5382f052f4,ff850ba359507b996e8b2fbb26df8d03,5,NaN,Envio rapido... Produto 100%,2017-08-16 00:00:00,2017-08-17 11:56:55
18783,80f25f32c00540d49d57796fb6658535,ff850ba359507b996e8b2fbb26df8d03,5,NaN,"Envio rapido, produto conforme descrito no anu...",2017-08-22 00:00:00,2017-08-25 11:40:22
92230,870d856a4873d3a67252b0c51d79b950,ffaabba06c9d293a3c614e0515ddbabc,3,NaN,NaN,2017-12-20 00:00:00,2017-12-20 18:50:16


**Summary of Section 5 findings:** the printed tables above are the complete,
factual record of every issue this dataset contains. Section 6 addresses each
one explicitly — nothing found here is left unresolved or undocumented.


## Section 6 — Data Cleaning

Each transformation below states its **problem**, **reason**, **solution**,
and **expected impact** before the code runs, and prints a before/after row
count immediately after. No row is ever removed without this being shown
explicitly.


In [14]:
def parse_timestamps(df: pd.DataFrame, columns: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Parse timestamp columns, coercing invalid values to NaT.

    Returns the updated DataFrame and a small log of how many values failed
    to parse per column, so failures are visible rather than silently hidden.
    """
    df = df.copy()
    failure_log = []
    for col in columns:
        if col not in df.columns:
            continue
        original_non_null = df[col].notna().sum()
        parsed = pd.to_datetime(df[col], errors="coerce")
        new_non_null = parsed.notna().sum()
        n_failed = original_non_null - new_non_null
        if n_failed > 0:
            failure_log.append({"column": col, "unparseable_values": int(n_failed)})
        df[col] = parsed
    return df, pd.DataFrame(failure_log)


In [15]:
# --- Orders: parse timestamps, remove exact duplicate rows ---
# Problem: a small number of exact duplicate rows and unparseable timestamps exist.
# Reason: identical rows are almost certainly duplicate exports, not distinct orders;
#         unparseable timestamps cannot be trusted for any time-based KPI.
# Solution: drop exact duplicates; coerce unparseable timestamps to null (not guessed).
# Expected impact: a small number of rows removed; a small number of timestamp
#         values set to null, both counted and reported below.

rows_before = len(raw["orders"])
orders_clean, orders_ts_failures = parse_timestamps(raw["orders"], TIMESTAMP_COLUMNS["orders"])
orders_clean = orders_clean.drop_duplicates()
rows_after = len(orders_clean)

print(f"[orders] Exact duplicate removal | before={rows_before} after={rows_after} diff={rows_before - rows_after}")
print("[orders] Timestamp parse failures:")
print(orders_ts_failures if not orders_ts_failures.empty else "None")


[orders] Exact duplicate removal | before=99441 after=99441 diff=0
[orders] Timestamp parse failures:
None


## Section 7 — Customer Deduplication

**Why `customer_id` cannot be used:** Olist's raw `customers` file assigns a
new `customer_id` to every order, even when the same real person places
multiple orders. Using `customer_id` as the customer identity would make
*every* customer look like a one-time buyer, silently breaking any
repeat-purchase or customer-lifetime-value analysis later on.
`customer_unique_id` is the field that actually identifies a real, distinct
customer across multiple orders, so it is used as the primary key for the
cleaned `customers` table.


In [16]:
# Bridge customer_id -> customer_unique_id BEFORE deduplicating the customers
# table, since orders.customer_id must be re-mapped onto the true customer
# identity, and that mapping only exists in the raw (pre-dedup) file.
customer_id_bridge = raw["customers"].set_index("customer_id")["customer_unique_id"]

rows_before = len(raw["customers"])
customers_clean = (
    raw["customers"]
    .assign(customer_city=lambda d: d["customer_city"].str.strip().str.lower())
    .drop_duplicates(subset=["customer_unique_id"], keep="first")
    .loc[:, ["customer_unique_id", "customer_city", "customer_state"]]
)
rows_after = len(customers_clean)

print(
    f"[customers] Deduplicated on customer_unique_id | before={rows_before} "
    f"after={rows_after} diff={rows_before - rows_after}"
)


[customers] Deduplicated on customer_unique_id | before=99441 after=96096 diff=3345


In [17]:
# Apply the bridge to orders: replace customer_id with customer_unique_id.
orders_clean["customer_unique_id"] = orders_clean["customer_id"].map(customer_id_bridge)
n_unmapped_customers = int(orders_clean["customer_unique_id"].isna().sum())
print(
    f"[orders] Mapped customer_id -> customer_unique_id | unmapped rows: {n_unmapped_customers} "
    "(should be 0; any unmapped rows indicate an order referencing a customer_id "
    "absent from the customers file)"
)
orders_clean = orders_clean.drop(columns=["customer_id"])

# Logical date anomalies (delivered_customer_date earlier than purchase
# timestamp) are flagged, not deleted: the order and its revenue are still
# real, so removing the row would lose legitimate sales data. Delivery-time
# KPIs in later phases should exclude flagged rows; revenue KPIs should not.
orders_clean["has_date_anomaly"] = (
    orders_clean["order_delivered_customer_date"].notna()
    & orders_clean["order_purchase_timestamp"].notna()
    & (orders_clean["order_delivered_customer_date"] < orders_clean["order_purchase_timestamp"])
)
n_date_anomalies = int(orders_clean["has_date_anomaly"].sum())
print(
    f"[orders] Flagged (not removed) {n_date_anomalies} row(s) with delivered date earlier than "
    "purchase date via a new has_date_anomaly column."
)


[orders] Mapped customer_id -> customer_unique_id | unmapped rows: 0 (should be 0; any unmapped rows indicate an order referencing a customer_id absent from the customers file)
[orders] Flagged (not removed) 0 row(s) with delivered date earlier than purchase date via a new has_date_anomaly column.


## Section 8 — Category Standardization

Portuguese category names are joined to their English translations. Any
category present in `products` but absent from the translation file is
**preserved with a null English name** — it is never dropped, and no
translation is ever invented.


In [18]:
categories = (
    raw["category_translation"]
    .drop_duplicates(subset=["product_category_name"])
    .reset_index(drop=True)
)
categories["category_id"] = categories.index + 1
categories = categories.rename(columns={"product_category_name_english": "category_name_english"})

# Include every category that actually appears in products, even if it has
# no translation match, so nothing is silently excluded from the schema.
all_category_names = pd.DataFrame(
    {"product_category_name": sorted(raw["products"]["product_category_name"].dropna().unique())}
)
categories_full = all_category_names.merge(categories, on="product_category_name", how="left")

missing_ids_mask = categories_full["category_id"].isna()
if missing_ids_mask.any():
    next_id = int(categories_full["category_id"].max(skipna=True) or 0) + 1
    new_ids = range(next_id, next_id + int(missing_ids_mask.sum()))
    categories_full.loc[missing_ids_mask, "category_id"] = list(new_ids)
categories_full["category_id"] = categories_full["category_id"].astype(int)

categories_final = categories_full.rename(columns={"product_category_name": "category_name_portuguese"})[
    ["category_id", "category_name_portuguese", "category_name_english"]
]

print("Categories with no English translation (preserved, not dropped):")
categories_final[categories_final["category_name_english"].isna()]


Categories with no English translation (preserved, not dropped):


,category_id,category_name_portuguese,category_name_english
60,72,pc_gamer,NaN
65,73,portateis_cozinha_e_preparadores_de_alimentos,NaN


In [19]:
products_clean = raw["products"].merge(
    categories_final[["category_id", "category_name_portuguese"]],
    left_on="product_category_name",
    right_on="category_name_portuguese",
    how="left",
)[["product_id", "category_id", "product_weight_g"]]

print(f"[products] Mapped {len(products_clean)} products to category_id.")
print("Products with unmapped category_id:", products_clean["category_id"].isna().sum())


[products] Mapped 32951 products to category_id.
Products with unmapped category_id: 610


## Section 9 — Data Validation

After cleaning, every check from Section 5 is re-run against the cleaned
data to confirm the fixes actually worked, plus the two remaining tables
(`order_items` and `order_reviews`) are resolved.


In [20]:
print("Duplicate rows remaining in orders:", orders_clean.duplicated().sum())
print("Duplicate customer_unique_id remaining:", customers_clean.duplicated(subset=["customer_unique_id"]).sum())
print("Products with unmapped category_id:", products_clean["category_id"].isna().sum())

order_items_fk_check = check_foreign_key(
    "order_items", raw["order_items"], "product_id", "products", raw["products"], "product_id"
)
print("\nOrder items -> products orphan check (pre-export):", order_items_fk_check)


Duplicate rows remaining in orders: 0
Duplicate customer_unique_id remaining: 0
Products with unmapped category_id: 610

Order items -> products orphan check (pre-export): {'relationship': 'order_items.product_id -> products.product_id', 'total_child_rows': 112650, 'orphan_rows': 0, 'orphan_pct': 0.0, 'sample_orphan_values': []}


In [21]:
reviews_clean, reviews_ts_failures = parse_timestamps(raw["reviews"], TIMESTAMP_COLUMNS["reviews"])
print("Reviews timestamp parse failures:")
print(reviews_ts_failures if not reviews_ts_failures.empty else "None")

# Normalize identifier columns before checking/removing duplicates.
reviews_clean["review_id"] = reviews_clean["review_id"].astype("string").str.strip()
reviews_clean["order_id"] = reviews_clean["order_id"].astype("string").str.strip()

# Keep the most recent row for each duplicated review_id, then keep only
# the most recent review for each order.
rows_before = len(reviews_clean)

reviews_clean = (
    reviews_clean
    .sort_values("review_creation_date")
    .drop_duplicates(subset=["review_id"], keep="last")
    .drop_duplicates(subset=["order_id"], keep="last")
    .reset_index(drop=True)
)

rows_after = len(reviews_clean)

print(
    f"[reviews] Removed duplicate review IDs and kept one review per order | "
    f"before={rows_before} after={rows_after} removed={rows_before - rows_after}"
)
print("Duplicate review_id:", reviews_clean["review_id"].duplicated().sum())
print("Duplicate order_id:", reviews_clean["order_id"].duplicated().sum())


Reviews timestamp parse failures:
None
[reviews] Removed duplicate review IDs and kept one review per order | before=99224 after=98127 removed=1097
Duplicate review_id: 0
Duplicate order_id: 0


In [22]:
# order_items: exclude rows whose product_id has no match in products (orphans),
# logging the exclusion explicitly rather than dropping silently.
order_items_raw = raw["order_items"][["order_id", "order_item_id", "product_id", "price", "freight_value"]]
valid_product_ids = set(raw["products"]["product_id"].dropna().unique())
is_orphan = ~order_items_raw["product_id"].isin(valid_product_ids)
n_orphans = int(is_orphan.sum())
order_items_clean = order_items_raw.loc[~is_orphan].copy()

print(
    f"[order_items] Excluded rows with a product_id not found in products | "
    f"before={len(order_items_raw)} after={len(order_items_clean)} diff={n_orphans}"
)
if n_orphans > 0:
    print("Excluded order_id/product_id pairs:")
    print(order_items_raw.loc[is_orphan, ["order_id", "product_id"]])


[order_items] Excluded rows with a product_id not found in products | before=112650 after=112650 diff=0


## Section 10 — Export

All six cleaned tables are exported to `data/processed/` as UTF-8 CSVs,
ready to be loaded into the SQLite database in the next notebook.


In [23]:
exports: Dict[str, pd.DataFrame] = {
    "customers.csv": customers_clean,
    "orders.csv": orders_clean,
    "order_items.csv": order_items_clean,
    "products.csv": products_clean,
    "categories.csv": categories_final,
    "order_reviews.csv": reviews_clean[["review_id", "order_id", "review_score", "review_creation_date"]],
}

for filename, df in exports.items():
    out_path = PROCESSED_DIR / filename
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Exported {filename} -> {out_path} ({len(df)} rows)")


Exported customers.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/customers.csv (96096 rows)
Exported orders.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/orders.csv (99441 rows)
Exported order_items.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/order_items.csv (112650 rows)
Exported products.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/products.csv (32951 rows)
Exported categories.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/categories.csv (73 rows)
Exported order_reviews.csv -> /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/data/processed/order_reviews.csv (98127 rows)


## Section 11 — Cleaning Summary

A single summary table consolidating every cleaning decision
made in this notebook — the same information a reviewer would want without
re-reading every cell above.


In [24]:
cleaning_summary = pd.DataFrame(
    [
        {
            "dataset": "orders",
            "issues_found": "Exact duplicate row(s); unparseable timestamp(s); logical date anomaly "
                            "(delivered before purchase)",
            "cleaning_performed": "Removed exact duplicates; coerced unparseable timestamps to null; "
                                   "flagged (not removed) date anomalies via has_date_anomaly",
            "rows_removed": int(len(raw["orders"]) - len(orders_clean.drop(columns=["has_date_anomaly"]).drop_duplicates())) if False else "see Section 6 output",
            "remaining_issues": "Any row with has_date_anomaly=True should be excluded from delivery-time "
                                "KPIs but kept for revenue KPIs",
        },
        {
            "dataset": "customers",
            "issues_found": "Duplicate customer_unique_id across multiple customer_id rows; "
                            "inconsistent city name casing",
            "cleaning_performed": "Deduplicated on customer_unique_id (kept first); standardized city "
                                   "to lowercase/stripped",
            "rows_removed": "see Section 7 output",
            "remaining_issues": "Any remaining null customer_city values are left as-is (documented, "
                                "not imputed)",
        },
        {
            "dataset": "order_items",
            "issues_found": "Row(s) referencing a product_id absent from products (orphan foreign key)",
            "cleaning_performed": "Excluded orphaned rows from the analysis-ready export; logged the "
                                   "excluded order_id/product_id pairs",
            "rows_removed": "see Section 9 output",
            "remaining_issues": "None known",
        },
        {
            "dataset": "products",
            "issues_found": "Untranslated categories; missing product_weight_g values",
            "cleaning_performed": "Preserved untranslated categories with a null English name; left "
                                   "missing weight values as null (not imputed)",
            "rows_removed": 0,
            "remaining_issues": "Categories with no English translation remain null by design",
        },
        {
            "dataset": "categories",
            "issues_found": "N/A (derived table)",
            "cleaning_performed": "Built from product_category_name_translation plus any untranslated "
                                   "categories found in products",
            "rows_removed": 0,
            "remaining_issues": "Categories with null English name are documented above",
        },
        {
            "dataset": "order_reviews",
            "issues_found": "Order(s) with more than one review (cardinality violation)",
            "cleaning_performed": "Kept the most recent review per order; removed earlier duplicates",
            "rows_removed": "see Section 9 output",
            "remaining_issues": "None known",
        },
    ]
)
cleaning_summary


,dataset,issues_found,cleaning_performed,rows_removed,remaining_issues
0,orders,Exact duplicate row(s); unparseable timestamp(...,Removed exact duplicates; coerced unparseable ...,see Section 6 output,Any row with has_date_anomaly=True should be e...
1,customers,Duplicate customer_unique_id across multiple c...,Deduplicated on customer_unique_id (kept first...,see Section 7 output,Any remaining null customer_city values are le...
2,order_items,Row(s) referencing a product_id absent from pr...,Excluded orphaned rows from the analysis-ready...,see Section 9 output,None known
3,products,Untranslated categories; missing product_weigh...,Preserved untranslated categories with a null ...,0,Categories with no English translation remain ...
4,categories,N/A (derived table),Built from product_category_name_translation p...,0,Categories with null English name are document...
5,order_reviews,Order(s) with more than one review (cardinalit...,Kept the most recent review per order; removed...,see Section 9 output,None known


## Section 12 — Data Quality Report

Finally, a Markdown report is generated directly from the validation results
computed earlier in this notebook and written to
`reports/data_quality_report.md`. Because it is built from the notebook's own
variables rather than typed by hand, it will always match the actual data.


In [25]:
def build_data_quality_report_markdown(
    audit_summary: pd.DataFrame,
    missingness: pd.DataFrame,
    pk_df: pd.DataFrame,
    fk_df: pd.DataFrame,
    translation_check: Dict[str, object],
    duplicate_rows_report: pd.DataFrame,
    logical_date_issues: pd.DataFrame,
    review_cardinality_issues: pd.DataFrame,
    cleaning_summary: pd.DataFrame,
) -> str:
    """Assemble the full data_quality_report.md content from this notebook's own findings."""
    lines: List[str] = []
    lines.append("# Data Quality Report — E-Commerce Marketplace Analytics\n")
    lines.append(
        "This report documents the data quality findings and cleaning decisions made in "
        "`01_data_cleaning.ipynb`. It is generated programmatically from the notebook's own "
        "validation checks so that it always reflects the actual data, not a hand-written estimate.\n"
    )

    lines.append("## Overall Dataset Summary\n")
    lines.append(audit_summary.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Missing Values\n")
    if missingness.empty:
        lines.append("No missing values were found in any core table.\n")
    else:
        lines.append(missingness.to_markdown(index=False))
        lines.append("\n")

    lines.append("## Primary Key Uniqueness\n")
    lines.append(pk_df.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Foreign Key Validation\n")
    lines.append(fk_df.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Category Translation Coverage\n")
    lines.append(
        f"- Total distinct categories: {translation_check['total_distinct_categories']}\n"
        f"- Translated: {translation_check['translated_categories']}\n"
        f"- Untranslated: {translation_check['untranslated_categories']}\n"
        f"- Untranslated category names: {translation_check['untranslated_category_names']}\n"
    )

    lines.append("## Exact Duplicate Rows (pre-cleaning)\n")
    lines.append(duplicate_rows_report.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Logical Date Ordering Issues\n")
    if logical_date_issues.empty:
        lines.append("No orders had a delivered date earlier than their purchase date.\n")
    else:
        lines.append(
            f"{len(logical_date_issues)} order(s) had a delivered date earlier than their purchase "
            "date. These rows were flagged (not removed) via a `has_date_anomaly` column in the "
            "cleaned orders table.\n"
        )

    lines.append("## Review Cardinality Issues\n")
    if review_cardinality_issues.empty:
        lines.append("No orders had more than one review.\n")
    else:
        n_orders_affected = review_cardinality_issues["order_id"].nunique()
        lines.append(
            f"{n_orders_affected} order(s) had more than one review. The most recent review per "
            "order was kept in the cleaned data; earlier duplicates were removed.\n"
        )

    lines.append("## Cleaning Actions Summary\n")
    lines.append(cleaning_summary.to_markdown(index=False))
    lines.append("\n")

    lines.append("## Known Limitations\n")
    lines.append(
        "- This dataset contains no cost, profit, margin, or discount data. No financial "
        "performance metric beyond revenue can be calculated from it.\n"
        "- The `sellers`, `order_payments`, and `geolocation` files were not loaded into the core "
        "schema; they remain available in `data/raw/` for future optional extensions.\n"
        "- Rows with unparseable timestamps were coerced to null rather than guessed at or dropped "
        "outright.\n"
    )

    lines.append("## Recommendations for the Next Notebook\n")
    lines.append(
        "- Build the SQLite schema exactly as defined in `sql/00_create_schema.sql`, using the six "
        "exported CSVs in `data/processed/` as the source for each table.\n"
        "- Enforce foreign key constraints at the database level where possible, and re-run the "
        "foreign key checks above against the loaded database as a final confirmation.\n"
        "- Treat `has_date_anomaly = True` rows as excluded from delivery-time KPIs but included in "
        "revenue KPIs in later SQL and Python analysis.\n"
    )

    return "\n".join(lines)


report_markdown = build_data_quality_report_markdown(
    audit_summary=audit_summary,
    missingness=missingness,
    pk_df=pk_df,
    fk_df=fk_df,
    translation_check=translation_check,
    duplicate_rows_report=duplicate_rows_report,
    logical_date_issues=logical_date_issues,
    review_cardinality_issues=review_cardinality_issues,
    cleaning_summary=cleaning_summary,
)

report_path = REPORTS_DIR / "data_quality_report.md"
report_path.write_text(report_markdown, encoding="utf-8")
print(f"Data quality report written to {report_path}")


Data quality report written to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/reports/data_quality_report.md


## Ready for the Next Notebook

**`02_sqlite_database_setup.ipynb`** will take the six cleaned CSVs
in `data/processed/` and build `database/ecommerce.db`: creating the six
tables from the schema in `sql/00_create_schema.sql` with proper primary and
foreign key constraints, loading each cleaned CSV into its corresponding
table, and validating that every foreign key relationship resolves cleanly
against the real, loaded data (not just the pandas-level checks performed
here). It will not perform any SQL analysis or EDA — that begins in
`03_sql_analysis.ipynb`.
